# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library. The dataset is described by a Croissant JSON-LD schema and includes multiple record sets and fields to analyze.

### Dataset Source
The dataset is specified by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Make sure the mlcroissant library is installed in your environment
!pip install mlcroissant

## 1. Data Loading

We'll load the dataset metadata and records from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(url)

# Print key metadata (using attributes, not dict subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")

# List available record sets in the metadata
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    print("\nRecord set(s) present in this dataset:")
    # recordSet can be a list or single dict
    from collections.abc import Sequence
    rs = dataset.metadata.recordSet
    if isinstance(rs, Sequence) and not isinstance(rs, str):
        for record_set in rs:
            print(f"  - {record_set.get('@id', str(record_set))}")
    else:
        print(f"  - {rs.get('@id', str(rs))}")
else:
    print("No explicit recordSet found in metadata.\nYou may still be able to iterate available record sets via dataset.record_sets().")

## 2. Data Overview

Examine all available record sets, fields, and their unique `@id` values, as well as a preview of each record set. All references will use `@id`.

In [ ]:
# List all record sets defined in the dataset

record_set_ids = list(dataset.record_sets())

print(f"Found {len(record_set_ids)} record set(s):")
for i, rid in enumerate(record_set_ids):
    print(f"{i+1}. @id: {rid}")

# Show available fields for each record set with their @id
for rid in record_set_ids:
    record_set = dataset.get_record_set(rid)
    print(f"\nRecord set '@id': {rid}")
    if hasattr(record_set, 'field') and record_set.field:
        fields = record_set.field
        if isinstance(fields, list):
            for field in fields:
                print(f"  - field @id: {field.get('@id', str(field))}")
        else:
            print(f"  - field @id: {fields.get('@id', str(fields))}")
    else:
        print("  (No fields found)")

# Preview a row from each record set
for rid in record_set_ids:
    print(f"\nFirst record from record set '@id': {rid}")
    try:
        rec_iter = dataset.records(record_set=rid)
        print(next(rec_iter))
    except StopIteration:
        print("  (No records found)")
    except Exception as e:
        print(f"  (Error reading records: {e})")

## 3. Data Extraction

We extract data from each available record set using their `@id`. All record set and field references use their Croissant `@id`, ensuring compatibility and traceability.

In [ ]:
dataframes = {}

# Load all record sets into pandas DataFrames using @id
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    dataframes[rid] = pd.DataFrame(records) if records else pd.DataFrame()
    print(f"Loaded {len(records)} records from record set '@id': {rid}")

# Choose a primary record set (first one) for demonstration, using its @id
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and not dataframes[main_record_set_id].empty:
    print(f"\nColumns in DataFrame for '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No data available in the identified record sets.")

## 4. Exploratory Data Analysis (EDA)

Apply standard EDA steps. Let's select a numeric field (by `@id`), apply filtering, normalization, and grouping.

In [ ]:
# For this demonstration, let's select a numeric column and group field if available
import numpy as np

if main_record_set_id and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    # Attempt to automatically select a numeric field/column, else fallback to an example
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        # Fallback: choose a likely numeric column name by heuristics
        numeric_field = None
        for col in df.columns:
            if any(k in col.lower() for k in ['age', 'interval', 'count', 'number', 'size', 'years', 'duration']):
                numeric_field = col
                break
    if numeric_field:
        # Filter on values greater than the mean (as threshold example)
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 1
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Records where '{numeric_field}' > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        norm_col = numeric_field + '_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field] - mean) / std if std != 0 else 0
        print(f"\nNormalized '{numeric_field}' values:")
        display(filtered_df[[numeric_field, norm_col]].head())
        # Pick a possible group-by field (heuristics)
        group_field = None
        for col in df.columns:
            if any(k in col.lower() for k in ['sex', 'gender', 'group', 'tumor', 'location', 'msi', 'site', 'type', 'status']) and col != numeric_field:
                group_field = col
                break
        if group_field is not None and group_field in filtered_df.columns:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped by '{group_field}' (mean of '{numeric_field}'):")
            display(grouped)
    else:
        print("No obvious numeric field identified in the main record set.")
else:
    print("Data unavailable for EDA.")

## 5. Visualization

Let's visualize the distribution of a numeric field and its grouping by another field.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id and not dataframes[main_record_set_id].empty and 'numeric_field' in locals() and numeric_field and numeric_field in df:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    df[numeric_field].dropna().hist(bins=15, color='cornflowerblue', edgecolor='black')
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group_field if found
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load, inspect, and analyze the FAIR<sup>2</sup> colorectal cancer survivors dataset described in a Croissant schema.

- All data elements (record sets, fields, columns) were referenced and manipulated using their unique `@id`.
- We explored dataset metadata, examined available record sets, extracted and explored their data, and demonstrated basic EDA, including filtering, normalization, grouping, and plot visualizations.
- This workflow enables reproducible and standards-based exploration of FAIR data packages for further biomedical or ML analysis.